# Segmentación de países por patrones productivos

In [534]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

Carga de datos

In [535]:
df_ag = pd.read_csv('datos/produccion_agricola.csv')
df_gan = pd.read_csv('datos/produccion_ganadera.csv')
df_pob = pd.read_csv('datos/poblacion_2011_2020.csv')

Creacion de variables agricolas

In [536]:
# Emisiones
df_ag['emisiones_agricolas_ton'] = ((df_ag['fertilizantes_kg_ha'] * df_ag['superficie_hectareas']) / 1000) * 3.5

# Agregamos superficie total y producción total por país/año para sacar la eficiencia
ag_resumen = df_ag.groupby(['pais', 'anio']).agg({
    'superficie_hectareas': 'sum',
    'produccion_ton': 'sum',
    'emisiones_agricolas_ton': 'sum'
}).reset_index()

ag_resumen['eficiencia_agricola_ton_ha'] = ag_resumen['produccion_ton'] / ag_resumen['superficie_hectareas']

# Pivotar Cultivos (Especialización)
df_ag_pivot = df_ag.pivot_table(index=['pais', 'anio'], columns='cultivo', 
                                values='produccion_ton', aggfunc='sum', fill_value=0).reset_index()

df_ag_final = pd.merge(df_ag_pivot, ag_resumen[['pais', 'anio', 'superficie_hectareas', 'eficiencia_agricola_ton_ha', 'emisiones_agricolas_ton']], on=['pais', 'anio'])

In [537]:
df_ag_final.head()

,pais,anio,Algodón,Arroz,Café,Caña de azúcar,Cebada,Girasol,Maíz,Soja,Trigo,Té,superficie_hectareas,eficiencia_agricola_ton_ha,emisiones_agricolas_ton
0,Alemania,2011,0,0,745221,374446753,2838516,46148,22672727,17310419,23847803,573983,18663182,23.708796,7.299549e+06
1,Alemania,2012,0,0,584593,242465865,3933606,16246391,52096663,24195335,10308998,380432,21931188,15.968669,1.531082e+07
2,Alemania,2013,0,0,422736,65698549,4865054,4386555,58852779,18623769,12552324,937549,19569992,8.499713,1.302271e+07
3,Alemania,2014,0,0,314329,320503951,6717294,5602407,10671889,5847509,9007544,148035,14111481,25.427023,1.109797e+07
4,Alemania,2015,0,0,800024,363043576,11198736,2151027,39734991,2675561,510303,64087,15707571,26.750050,6.124890e+06


Cración de variabales ganaderas

In [538]:
# Promediamos la eficiencia de carne y sumamos producciones
gan_resumen = df_gan.groupby(['pais', 'anio']).agg({
    'eficiencia_carne': 'mean',
    'emisiones_ch4_ton_co2eq': 'sum'
}).reset_index()

# Pivotar Carne y Leche
df_gan_pivot = df_gan.pivot_table(index=['pais', 'anio'], columns='tipo_ganado', 
                                  values=['produccion_carne_ton', 'produccion_leche_lt'], aggfunc='sum', fill_value=0)
df_gan_pivot.columns = [f"{col[0].replace('produccion_', '').replace('_ton', '').replace('_lt', '')}_{col[1]}" for col in df_gan_pivot.columns]
df_gan_pivot = df_gan_pivot.reset_index()

df_gan_final = pd.merge(df_gan_pivot, gan_resumen, on=['pais', 'anio'])

In [539]:
df_gan_final.head()

,pais,anio,carne_avicola,carne_bovino,carne_caprino,carne_ovino,carne_porcino,leche_avicola,leche_bovino,leche_caprino,leche_ovino,leche_porcino,eficiencia_carne,emisiones_ch4_ton_co2eq
0,Alemania,2011,1047901,2531344,441906,147139,2109350,0,7990713373,158150093,447045971,0,0.63852,64352378
1,Alemania,2012,1128839,591490,417204,172779,941168,0,9615157014,254172201,358070905,0,0.41634,65989492
2,Alemania,2013,1048639,1322886,159189,242398,2336057,0,9037295589,496450828,289963119,0,0.37728,23695748
3,Alemania,2014,1530102,1005535,203491,188835,2766746,0,1724338309,369183291,186845629,0,0.11860,70826767
4,Alemania,2015,1205461,905516,466203,34476,2221576,0,5991702620,387007128,614568777,0,0.07112,56490776


Combinar datasets

In [540]:
df_merged = pd.merge(df_ag_final, df_gan_final, on=['pais', 'anio'], how='inner')
df_final = pd.merge(df_merged, df_pob, on=['pais', 'anio'], how='inner')

In [541]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   pais                        120 non-null    object 
 1   anio                        120 non-null    int64  
 2   Algodón                     120 non-null    int64  
 3   Arroz                       120 non-null    int64  
 4   Café                        120 non-null    int64  
 5   Caña de azúcar              120 non-null    int64  
 6   Cebada                      120 non-null    int64  
 7   Girasol                     120 non-null    int64  
 8   Maíz                        120 non-null    int64  
 9   Soja                        120 non-null    int64  
 10  Trigo                       120 non-null    int64  
 11  Té                          120 non-null    int64  
 12  superficie_hectareas        120 non-null    int64  
 13  eficiencia_agricola_ton_ha  120 non

Calculo de producción per capita para cultivos, carne y leche

In [542]:
cultivos = df_ag_pivot.columns.drop(['pais', 'anio'])
productos_ganado = df_gan_pivot.columns.drop(['pais', 'anio'])

# Producciones per cápita
for col in list(cultivos) + list(productos_ganado):
    df_final[f'{col}_per_capita'] = df_final[col] / df_final['poblacion']
    

In [543]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   pais                        120 non-null    object 
 1   anio                        120 non-null    int64  
 2   Algodón                     120 non-null    int64  
 3   Arroz                       120 non-null    int64  
 4   Café                        120 non-null    int64  
 5   Caña de azúcar              120 non-null    int64  
 6   Cebada                      120 non-null    int64  
 7   Girasol                     120 non-null    int64  
 8   Maíz                        120 non-null    int64  
 9   Soja                        120 non-null    int64  
 10  Trigo                       120 non-null    int64  
 11  Té                          120 non-null    int64  
 12  superficie_hectareas        120 non-null    int64  
 13  eficiencia_agricola_ton_ha  120 non

Agrupar por pais para hacer la media de cada país, sino los clusters podrían llevar a que España 2011 estuviera en uno y España 2015 en otro diferente.

In [544]:
# Variables finales: Eficiencias + todas las per cápita
cols_modelo = [col for col in df_final.columns if 'per_capita' in col] + ['eficiencia_agricola_ton_ha', 'eficiencia_carne','emisiones_agricolas_ton', 'emisiones_ch4_ton_co2eq', 'superficie_hectareas']
df_modelo = df_final.groupby('pais')[cols_modelo].mean().reset_index()

In [545]:
df_modelo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   pais                        12 non-null     object 
 1   Algodón_per_capita          12 non-null     float64
 2   Arroz_per_capita            12 non-null     float64
 3   Café_per_capita             12 non-null     float64
 4   Caña de azúcar_per_capita   12 non-null     float64
 5   Cebada_per_capita           12 non-null     float64
 6   Girasol_per_capita          12 non-null     float64
 7   Maíz_per_capita             12 non-null     float64
 8   Soja_per_capita             12 non-null     float64
 9   Trigo_per_capita            12 non-null     float64
 10  Té_per_capita               12 non-null     float64
 11  carne_avicola_per_capita    12 non-null     float64
 12  carne_bovino_per_capita     12 non-null     float64
 13  carne_caprino_per_capita    12 non-nu

Modelos

In [546]:
X = df_modelo.drop('pais', axis=1)
X_scaled = StandardScaler().fit_transform(X)

In [547]:
# K-means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_modelo['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

c:\Users\Ccp0897\anaconda3\envs\piaentorno\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [548]:
#DBSCAN
dbscan = DBSCAN(eps=2.5, min_samples=3) 
df_modelo['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

Evaluacion

In [549]:
def evaluar_clustering(X, labels, nombre):
    # Si DBSCAN solo encuentra ruido (-1) o 1 solo cluster, no se puede evaluar bien
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    
    if n_clusters > 1:
        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)
        print(f"--- {nombre} ---")
        print(f"Número de clústeres: {n_clusters}")
        print(f"Silhouette Score (Mejor cerca de 1): {sil:.4f}")
        print(f"Davies-Bouldin Index (Mejor cerca de 0): {db:.4f}")
        if -1 in labels:
            print(f"Países detectados como 'Ruido' (Atípicos): {list(labels).count(-1)}")
    else:
        print(f"--- {nombre} ---")
        print("El modelo no encontró suficientes clústeres para ser evaluado.")

evaluar_clustering(X_scaled, df_modelo['cluster_kmeans'], "K-MEANS")
print("\n")
evaluar_clustering(X_scaled, df_modelo['cluster_dbscan'], "DBSCAN")

--- K-MEANS ---
Número de clústeres: 3
Silhouette Score (Mejor cerca de 1): 0.2855
Davies-Bouldin Index (Mejor cerca de 0): 0.3741


--- DBSCAN ---
Número de clústeres: 2
Silhouette Score (Mejor cerca de 1): 0.0577
Davies-Bouldin Index (Mejor cerca de 0): 1.8042


Visualización

In [550]:
# 1. Ver qué países han caído en cada clúster
print("--- COMPOSICIÓN DE LOS CLÚSTERES ---")
for i in range(kmeans.n_clusters):
    paises = df_modelo[df_modelo['cluster_kmeans'] == i]['pais'].tolist()
    print(f"\nClúster {i}:")
    print(f"  Países: {', '.join(paises)}")

# 2. Ver las características promedio de cada clúster (para saber qué nombre ponerles)
# Seleccionamos las columnas clave para no saturar
columnas_clave = ['eficiencia_agricola_ton_ha', 'emisiones_ch4_ton_co2eq', 'superficie_hectareas']
resumen = df_modelo.groupby('cluster_kmeans')[columnas_clave].mean()
print("\n--- CARACTERÍSTICAS PROMEDIO POR CLÚSTER ---")
print(resumen)

--- COMPOSICIÓN DE LOS CLÚSTERES ---

Clúster 0:
  Países: Alemania, Argentina, Brasil, China, España, Estados Unidos, Francia, India, Kenia, México

Clúster 1:
  Países: Nueva Zelanda

Clúster 2:
  Países: Australia

--- CARACTERÍSTICAS PROMEDIO POR CLÚSTER ---
                eficiencia_agricola_ton_ha  emisiones_ch4_ton_co2eq  \
cluster_kmeans                                                        
0                                22.840044              62446025.32   
1                                17.548402              63480504.40   
2                                29.391992              64907099.30   

                superficie_hectareas  
cluster_kmeans                        
0                        18492935.87  
1                        17973464.30  
2                        21295292.60  
